In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from amms.core import products
from amms.core.analysis.sfr import lookback_edges, sfh

DATASET, R_MAX, DT = "ext_b12-model2", 5.0, 0.1 # kpc, Gyr

In [ ]:
def load_sfh(galaxy, snap=69, r_max=R_MAX, dt=DT):
    p = products.read_product(products.product_path(DATASET, "newstars", f"{galaxy}-{snap:03d}"))
    t_now = p.attrs.meta["t_now_gyr"]
    edges = lookback_edges(t_now, dt)
    pos, t, m = p.arrays["pos_kpc"], p.arrays["t_form_gyr"], p.arrays["mass_msun"]
    sel = np.linalg.norm(pos, axis=1) < r_max # position is already in the galaxy's own frame
    _, rate = sfh(t[sel], m[sel], bins=edges)
    _, rate_all = sfh(t, m, bins=edges)
    return p, t_now, edges, rate, rate_all

In [ ]:
p, t_now, edges, rate, rate_all = load_sfh("lmc")
print(p.attrs.meta["gadget_time_to_gyr"], "t_now", t_now, "dropped ft==0", p.attrs.meta["n_dropped_ft0"])
print("PIN all LMC stars, last 100 Myr:", rate_all[-1], "(0.0488 with the old 0.97)")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
# reversed so staris get increasing lookback edges with 0 as the present day
ax.stairs(rate[::-1], (t_now - edges)[::-1], label=f"r < {R_MAX:g} kpc")
ax.stairs(rate_all[::-1], (t_now - edges)[::-1], ls="--", label="all LMC")
ax.set(xlabel="lookback time [Gyr]", ylabel=r"SFR [$M_\odot\,\mathrm{yr}^{-1}$]")
ax.legend()